# E-commerce Data Engineering Lab

In this notebook I take **50,000 sales orders** and walk them through the 12-step data engineering road-map: **ingest → wrangle → clean → transform → feature-engineer → aggregate → serialize**. It ends with one analytical insight.

**Data used**

| Role | File | Source |
|---|---|---|
| Primary sales orders | `data/raw/sales_records_dirty.csv` (50,250 rows) | The [ExcelBIAnalytics "50000 Sales Records"](https://excelbianalytics.com/wp/downloads-18-sample-csv-files-data-sets-for-testing-sales/) sample file (kept untouched as `data/raw/sales_records_50k_source.csv`), with realistic grime added by [`scripts/add_grime.py`](scripts/add_grime.py) using a fixed seed. |
| Secondary metadata | `data/raw/world_gdp_2014.csv` (222 rows) | Plotly's open [2014 world GDP](https://github.com/plotly/datasets/blob/master/2014_world_gdp_with_codes.csv) dataset: country name, GDP in billions of USD, and ISO country code. |

**Why add grime?** The downloaded file is almost clean. Its only natural problem is trailing spaces in 1,973 country names. Real order exports are messier, so the script adds the kinds of problems a real export would have (a second date format, `$` prices, blanks, spelling variants, a double export). The seed is fixed, so every run finds exactly the same problems. Because I still have the untouched source, Step 9 can check the cleaned result against it.

I use the secondary file for two things: to **enrich** each order with its country's GDP and ISO code, and as the second source for the **Data Dictionary** at the end.

## Setup

All file paths are defined once here. The notebook uses paths relative to the repository root, so it runs the same after a fresh `git clone`.

In [1]:
import gzip
import json
from collections import Counter, defaultdict, namedtuple
from dataclasses import asdict, astuple, dataclass
from datetime import date, datetime
from pathlib import Path

import pandas as pd

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")

RAW_SALES = RAW_DIR / "sales_records_dirty.csv"
SOURCE_SALES = RAW_DIR / "sales_records_50k_source.csv"
RAW_GDP = RAW_DIR / "world_gdp_2014.csv"
CLEAN_CSV = PROCESSED_DIR / "sales_clean.csv"
CLEAN_JSON = PROCESSED_DIR / "sales_clean.json.gz"
PROFIT_JSON = PROCESSED_DIR / "profit_by_item_type.json"

## Step 1 — Hello, Data!

I read every column **as text** (`dtype=str`) and turn off pandas' automatic missing-value detection (`keep_default_na=False`). I want to see the file exactly as it was written.

If I let pandas parse it, `Unit Price` would silently become text because of the `$` signs, blank `Units Sold` would become `NaN` and turn the whole column into floats, and the two date formats would be hidden inside generic strings. The grime would be half-hidden before I had even looked at it.

In [2]:
raw_df = pd.read_csv(RAW_SALES, dtype=str, keep_default_na=False)

print(f"Loaded {len(raw_df):,} rows x {raw_df.shape[1]} columns")
raw_df.head(3)

Loaded 50,250 rows x 14 columns


,Region,Country,Item Type,Sales Channel,Order Priority,Order Date,Order ID,Ship Date,Units Sold,Unit Price,Unit Cost,Total Revenue,Total Cost,Total Profit
0,Sub-Saharan Africa,Namibia,Household,Offline,M,2015-08-31,897751939,2015-10-12,3604,668.27,502.54,2408445.08,1811154.16,597290.92
1,Europe,Iceland,Baby Food,Online,H,11/20/2010,599480426,1/9/2011,8435,255.28,159.42,2153286.80,1344707.70,808579.10
2,Europe,Russia,Meat,Online,L,6/22/2017,538911855,6/25/2017,4848,$421.89,364.69,2045322.72,1768017.12,277305.60


## Step 2 — Pick the Right Container

| Container | Where I use it | Why |
|---|---|---|
| **Class** (`@dataclass`) | One order: `SalesOrder` | An order must change during cleaning and carry behaviour (`clean()`, `revenue()`, `profit()`, `ship_days()`). A namedtuple is immutable, and a dict cannot hold methods. |
| **namedtuple** | `CountryInfo`, `PriceProfile` | Small, fixed, read-only records. A namedtuple gives named fields (`info.gdp_billions`) with no extra code, and its immutability protects reference data. |
| **dict** | `country_lookup`: country → `CountryInfo`; spelling maps; `region_by_country` | Enrichment means "look up by key" 50,000 times, and a dict does that in constant time. A list search would be 222 times slower per lookup. |
| **set** | Unique countries, unique order IDs, duplicate detection | A set removes duplicates automatically, so `len(set(...))` is the unique count, and membership checks are constant time. |

## Step 3 — Implement Functions and Data Structure

**The functions.** Each helper does one conversion and returns `None` when a value cannot be used. The helpers never guess.

Each one also **returns non-string input unchanged**. That makes cleaning *idempotent*: running `clean()` on an already-clean record changes nothing. Step 7 relies on this to measure the raw and cleaned data with the same code.

In [3]:
DATE_FORMATS = ("%m/%d/%Y", "%Y-%m-%d")          # the source's US format first; ISO seen in the raw file
PRIORITY_CODES = {"C", "H", "M", "L"}
MAX_UNITS = 10_000                               # the largest order line in the source file
UNKNOWN_REGION = "UNKNOWN"


def collapse_spaces(value: str) -> str:
    return " ".join(value.split())


def canonical_spellings(values: pd.Series) -> dict[str, str]:
    """Map each case-insensitive name to its most common spelling in the data."""
    spellings = {}
    for spelling, _ in Counter(values.map(collapse_spaces)).most_common():
        spellings.setdefault(spelling.casefold(), spelling)
    return spellings


def normalize_name(value: str | None, spellings: dict[str, str]) -> str | None:
    """'  south korea ' -> 'South Korea', using the most common spelling as the correct one."""
    if not isinstance(value, str):
        return value
    text = collapse_spaces(value)
    return spellings.get(text.casefold(), text)


def normalize_priority(value: str | None) -> str | None:
    """'High', 'high' or 'h' -> 'H'. Anything that is not C/H/M/L -> None."""
    if not isinstance(value, str):
        return value
    code = value.strip()[:1].upper()
    return code if code in PRIORITY_CODES else None


def parse_price(value: str | float | None) -> float | None:
    """'$421.89' -> 421.89. Returns None when the text is not a number."""
    if not isinstance(value, str):
        return value
    try:
        return float(value.replace("$", "").strip())
    except ValueError:
        return None


def parse_units(value: str | int | None) -> int | None:
    """'3604' -> 3604. Blank or non-numeric text -> None."""
    if not isinstance(value, str):
        return value
    text = value.strip()
    return int(text) if text.isdigit() else None


def parse_date(value: str | date | None) -> date | None:
    """Accepts '8/31/2015' or '2015-08-31'. Anything else -> None."""
    if not isinstance(value, str):
        return value
    for date_format in DATE_FORMATS:
        try:
            return datetime.strptime(value.strip(), date_format).date()
        except ValueError:
            continue
    return None

**Spelling maps.** A country name cannot simply be title-cased: `"cote d'ivoire".title()` gives `Cote D'Ivoire`, and `"bosnia and herzegovina".title()` gives `Bosnia And Herzegovina`. Instead, I compare names case-insensitively and use **the most common spelling in the data** as the correct one. The data votes, and the correct spelling wins by a wide margin because only a small share of rows is misspelled.

In [4]:
COUNTRY_SPELLINGS = canonical_spellings(raw_df["Country"])
ITEM_SPELLINGS = canonical_spellings(raw_df["Item Type"])
CHANNEL_SPELLINGS = canonical_spellings(raw_df["Sales Channel"])

for column, spellings in [("Country", COUNTRY_SPELLINGS), ("Item Type", ITEM_SPELLINGS), ("Sales Channel", CHANNEL_SPELLINGS)]:
    print(f"{column:13} raw spellings: {raw_df[column].nunique():4}  ->  real values: {len(spellings)}")

Country       raw spellings:  768  ->  real values: 185
Item Type     raw spellings:   60  ->  real values: 12
Sales Channel raw spellings:   10  ->  real values: 2


**The data structure.** `SalesOrder` is one order line.

- It starts out holding the raw text.
- `clean()` returns a **new**, typed `SalesOrder` rather than changing itself, so the raw version stays available for before/after comparisons.
- `problems()` lists everything that makes the record unusable.
- `revenue()`, `cost()` and `profit()` are **recalculated** from units and prices. The file's stored totals are not trusted (Step 6 shows why).

`OrderBook` is a collection of orders. It has its own batch-level `clean()`, which also removes duplicates and fills blank regions (that needs *other* rows, so it cannot happen inside a single order), plus revenue and profit totals.

In [5]:
COLUMN_TO_FIELD = {
    "Order ID": "order_id",
    "Order Date": "order_date",
    "Ship Date": "ship_date",
    "Region": "region",
    "Country": "country",
    "Item Type": "item_type",
    "Sales Channel": "sales_channel",
    "Order Priority": "order_priority",
    "Units Sold": "units_sold",
    "Unit Price": "unit_price",
    "Unit Cost": "unit_cost",
}


@dataclass
class SalesOrder:
    """One order line. Holds raw text until clean() returns a typed copy."""

    order_id: str
    order_date: str | date | None
    ship_date: str | date | None
    region: str
    country: str
    item_type: str
    sales_channel: str
    order_priority: str | None
    units_sold: str | int | None
    unit_price: str | float | None
    unit_cost: str | float | None

    @classmethod
    def from_row(cls, row: dict) -> "SalesOrder":
        """Build from one CSV row (a dict). The stored totals are left out on purpose."""
        return cls(**{field: row[column] for column, field in COLUMN_TO_FIELD.items()})

    def clean(self) -> "SalesOrder":
        return SalesOrder(
            order_id=self.order_id.strip(),
            order_date=parse_date(self.order_date),
            ship_date=parse_date(self.ship_date),
            region=self.region.strip(),
            country=normalize_name(self.country, COUNTRY_SPELLINGS),
            item_type=normalize_name(self.item_type, ITEM_SPELLINGS),
            sales_channel=normalize_name(self.sales_channel, CHANNEL_SPELLINGS),
            order_priority=normalize_priority(self.order_priority),
            units_sold=parse_units(self.units_sold),
            unit_price=parse_price(self.unit_price),
            unit_cost=parse_price(self.unit_cost),
        )

    def problems(self) -> list[str]:
        """Reasons this (cleaned) record cannot be used; an empty list means it is valid."""
        issues = []
        if self.order_date is None or self.ship_date is None:
            issues.append("unreadable date")
        elif self.ship_date < self.order_date:
            issues.append("shipped before it was ordered")
        if self.unit_price is None or self.unit_price <= 0:
            issues.append("missing or non-positive price")
        if self.unit_cost is None or self.unit_cost <= 0:
            issues.append("missing or non-positive cost")
        if self.units_sold is None:
            issues.append("missing units")
        elif not 1 <= self.units_sold <= MAX_UNITS:
            issues.append(f"units outside 1-{MAX_UNITS:,}")
        if self.order_priority is None:
            issues.append("unknown priority")
        return issues

    def revenue(self) -> float:
        return round(self.units_sold * self.unit_price, 2)

    def cost(self) -> float:
        return round(self.units_sold * self.unit_cost, 2)

    def profit(self) -> float:
        return round(self.revenue() - self.cost(), 2)

    def ship_days(self) -> int:
        return (self.ship_date - self.order_date).days


class OrderBook:
    """A collection of orders with batch-level cleaning, revenue and profit."""

    def __init__(self, orders: list[SalesOrder]) -> None:
        self.orders = orders

    def __len__(self) -> int:
        return len(self.orders)

    def clean(self) -> tuple["OrderBook", dict]:
        """Clean every record, drop duplicates, fill blank regions, drop invalid rows; return the result and a report."""
        cleaned = [order.clean() for order in self.orders]

        # Deduplicate on the *cleaned* values, so the same order typed two ways still counts once.
        seen, unique = set(), []
        for order in cleaned:
            key = astuple(order)
            if key not in seen:
                seen.add(key)
                unique.append(order)

        # A country always belongs to one region, so the other rows tell us a blank region.
        region_by_country = {order.country: order.region for order in unique if order.region}
        blank_regions = [order for order in unique if not order.region]
        for order in blank_regions:
            order.region = region_by_country.get(order.country, UNKNOWN_REGION)

        valid = [order for order in unique if not order.problems()]
        reasons = Counter(issue for order in unique for issue in order.problems())

        report = {
            "rows before": len(cleaned),
            "exact duplicates removed": len(cleaned) - len(unique),
            "blank regions filled from country": len(blank_regions),
            "invalid rows removed": len(unique) - len(valid),
            "rows after": len(valid),
            **{f"  reason: {reason}": count for reason, count in reasons.items()},
        }
        return OrderBook(valid), report

    def total_revenue(self) -> float:
        return round(sum(order.revenue() for order in self.orders), 2)

    def total_profit(self) -> float:
        return round(sum(order.profit() for order in self.orders), 2)

    def to_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame([asdict(order) for order in self.orders])

**Using it.** I populate a list with the first three rows, then clean each one. Row 1's dates are in ISO format and row 3's price has a `$` sign. After cleaning, both become the same types as every other row.

In [6]:
sample = [SalesOrder.from_row(row) for row in raw_df.head(3).to_dict(orient="records")]

for raw in sample:
    clean = raw.clean()
    print(f"{raw.order_id}: raw price {raw.unit_price!r:10} date {raw.order_date!r:13}"
          f"-> {clean.order_date}, {clean.units_sold:,} x {clean.unit_price} = revenue {clean.revenue():,.2f}, profit {clean.profit():,.2f}")

897751939: raw price '668.27'   date '2015-08-31' -> 2015-08-31, 3,604 x 668.27 = revenue 2,408,445.08, profit 597,290.92
599480426: raw price '255.28'   date '11/20/2010' -> 2010-11-20, 8,435 x 255.28 = revenue 2,153,286.80, profit 808,579.10
538911855: raw price '$421.89'  date '6/22/2017'  -> 2017-06-22, 4,848 x 421.89 = revenue 2,045,322.72, profit 277,305.60


## Step 4 — Bulk Loaded

Now I load everything. `DataFrame.to_dict(orient="records")` turns each row into a dict, and each dict becomes a `SalesOrder` in an `OrderBook`.

The GDP file is mapped into a **dict**: `country → CountryInfo`. Before trusting a plain dict, I check that no country appears twice in the file. Otherwise the dict would silently keep whichever row came last.

In [7]:
CountryInfo = namedtuple("CountryInfo", ["code", "gdp_billions"])

raw_book = OrderBook([SalesOrder.from_row(row) for row in raw_df.to_dict(orient="records")])

gdp_df = pd.read_csv(RAW_GDP)
assert gdp_df["COUNTRY"].is_unique, "a duplicated country would be silently overwritten in the dict"
country_lookup: dict[str, CountryInfo] = {
    row["COUNTRY"]: CountryInfo(row["CODE"], row["GDP (BILLIONS)"]) for row in gdp_df.to_dict(orient="records")
}

print(f"OrderBook: {len(raw_book):,} orders")
print(f"country_lookup: {len(country_lookup)} countries")
print("country_lookup['Germany'] =", country_lookup["Germany"])

OrderBook: 50,250 orders
country_lookup: 222 countries
country_lookup['Germany'] = CountryInfo(code='DEU', gdp_billions=3820.0)


## Step 5 — Quick Profiling

This is a first look at the numbers **before** any cleaning, taking the file at face value. `PriceProfile` is a namedtuple, so the result reads like a small report.

In [8]:
PriceProfile = namedtuple("PriceProfile", ["min", "mean", "max", "not_a_number"])


def profile_prices(raw_prices: pd.Series) -> PriceProfile:
    numbers = pd.to_numeric(raw_prices, errors="coerce")   # '$421.89' cannot be read -> NaN
    return PriceProfile(
        min=float(numbers.min()),
        mean=round(float(numbers.mean()), 2),
        max=float(numbers.max()),
        not_a_number=int(numbers.isna().sum()),
    )


price_profile = profile_prices(raw_df["Unit Price"])
raw_countries = set(raw_df["Country"])
raw_order_ids = set(raw_df["Order ID"])

print(price_profile)
print(f"Rows: {len(raw_df):,}   unique Order IDs: {len(raw_order_ids):,}")
print(f"Unique country values:  {len(raw_countries)}")
print(f"  after normalize_name: {len({normalize_name(c, COUNTRY_SPELLINGS) for c in raw_countries})}")
print(f"Unique priority values: {sorted(set(raw_df['Order Priority']))}")

PriceProfile(min=-668.27, mean=265.14, max=668.27, not_a_number=902)
Rows: 50,250   unique Order IDs: 50,000
Unique country values:  768
  after normalize_name: 185
Unique priority values: ['C', 'Critical', 'H', 'High', 'L', 'Low', 'M', 'Medium', 'c', 'critical', 'h', 'high', 'l', 'low', 'm', 'medium']


The profile already points at four problems:

- **The minimum price is negative.** A product cannot cost less than nothing.
- **Some prices are not numbers at all** (`not_a_number`), so the mean is being calculated on fewer rows than the file contains.
- **There are more rows than Order IDs**, so some orders appear more than once.
- **The country set is inflated.** The raw values contain several hundred "countries", which collapse to 185 once the spelling is normalised. Priority should be four one-letter codes but arrives in many forms.

## Step 6 — Spot the Grime

I turn each suspicion into a check that counts the affected rows and shows one example. I use `repr()` for the examples so that stray spaces are visible.

In [9]:
def find_grime(df: pd.DataFrame) -> pd.DataFrame:
    """Count each kind of dirty value in the raw, all-text sales file."""
    units = pd.to_numeric(df["Units Sold"], errors="coerce")
    price = pd.to_numeric(df["Unit Price"].str.replace("$", "", regex=False), errors="coerce")
    stored_revenue = pd.to_numeric(df["Total Revenue"], errors="coerce")
    order_date = df["Order Date"].map(parse_date)
    ship_date = df["Ship Date"].map(parse_date)

    def misspelled(column: str, spellings: dict[str, str]) -> pd.Series:
        return df[column].ne(df[column].map(lambda value: normalize_name(value, spellings)))

    checks = {
        "exact duplicate row":                   ("Order ID", df.duplicated()),
        "price stored with a $ sign":            ("Unit Price", df["Unit Price"].str.startswith("$")),
        "negative price":                        ("Unit Price", df["Unit Price"].str.startswith("-")),
        "blank units sold":                      ("Units Sold", df["Units Sold"].str.strip().eq("")),
        f"units above {MAX_UNITS:,}":            ("Units Sold", units.gt(MAX_UNITS)),
        "blank region":                          ("Region", df["Region"].str.strip().eq("")),
        "date in ISO instead of M/D/YYYY":       ("Order Date", df["Order Date"].str.fullmatch(r"\d{4}-\d{2}-\d{2}")),
        "ship date before order date":           ("Ship Date", ship_date.lt(order_date)),
        "country with odd case or spacing":      ("Country", misspelled("Country", COUNTRY_SPELLINGS)),
        "item type with odd case or spacing":    ("Item Type", misspelled("Item Type", ITEM_SPELLINGS)),
        "sales channel with odd case or spacing": ("Sales Channel", misspelled("Sales Channel", CHANNEL_SPELLINGS)),
        "priority not a C/H/M/L code":           ("Order Priority", ~df["Order Priority"].isin(PRIORITY_CODES)),
        "blank stored Total Revenue":            ("Total Revenue", df["Total Revenue"].str.strip().eq("")),
        "stored Total Revenue != units x price": ("Total Revenue", (stored_revenue - units * price).abs().gt(0.01)),
    }
    rows = []
    for issue, (column, mask) in checks.items():
        example = repr(df.loc[mask, column].iloc[0]) if mask.any() else ""
        rows.append({"issue": issue, "column": column, "rows affected": int(mask.sum()), "example": example})
    return pd.DataFrame(rows)


grime = find_grime(raw_df)
grime

,issue,column,rows affected,example
0,exact duplicate row,Order ID,250,'647978627'
1,price stored with a $ sign,Unit Price,902,'$421.89'
2,negative price,Unit Price,41,'-9.33'
3,blank units sold,Units Sold,120,''
4,"units above 10,000",Units Sold,25,'99999'
5,blank region,Region,152,''
6,date in ISO instead of M/D/YYYY,Order Date,2518,'2015-08-31'
7,ship date before order date,Ship Date,59,'12/29/2015'
8,country with odd case or spacing,Country,3135,'Moldova '
9,item type with odd case or spacing,Item Type,605,'FRUITS'


The **country with odd case or spacing** count includes the **1,973 trailing spaces that were already in the downloaded file** (e.g. `'Samoa '`), not just the injected variants. Grime is not only a classroom invention.

The secondary file has grime of its own that matters for the join: **the two sources name some countries differently.**

In [10]:
our_countries = {normalize_name(country, COUNTRY_SPELLINGS) for country in raw_countries}
not_in_gdp = sorted(our_countries - set(country_lookup))

print(f"Countries in the sales data: {len(our_countries)}")
print(f"...with no exact match in the GDP file: {len(not_in_gdp)}")
not_in_gdp

Countries in the sales data: 185
...with no exact match in the GDP file: 13


['Cape Verde',
 'Democratic Republic of the Congo',
 'East Timor',
 'Federated States of Micronesia',
 'Myanmar',
 'Nauru',
 'North Korea',
 'Republic of the Congo',
 'South Korea',
 'The Bahamas',
 'The Gambia',
 'United States of America',
 'Vatican City']

**What each problem would break if I ignored it:**

1. **Duplicates** would double-count revenue and profit for those orders.
2. **`$` prices** cannot be read as numbers. **Negative prices** would subtract revenue. Neither is a real sale price.
3. **Blank and 99,999 unit counts** mean I cannot know what was actually sold. 99,999 is ten times the largest real order.
4. **Mixed date formats** break any date arithmetic, and a **ship date before the order date** is impossible. One of the two dates is wrong, and I cannot tell which.
5. **Spelling variants** of countries, item types, channels and priorities split one real value into several groups, which corrupts every "per country" or "per item" result.
6. **Blank regions** would drop orders out of every regional total.
7. **Stored totals that are blank or 10× too high** show that the file's `Total Revenue` column cannot be trusted. Totals must be recalculated from units × price.
8. **Country names that differ between the two files** would silently leave those orders without GDP after the join (Step 8 shows this).